# IsoME – Real-Axis Eliashberg Solver, step by step

This notebook reproduces the functionality of `RealAxisSolver(inp)` from the IsoME package,
but splits it into its individual steps so that each one can be inspected and debugged separately:

1. **Read in of the inputs** (`checkInput!`, `InputParser!`)
2. **Calculation of the kernels** (`precompute` → `kernels`)
3. **Evaluation of the Migdal-Eliashberg equations on the real axis** (`solve_realAxis_cDOS` / `solve_realAxis_vDOS`)

No new functions are defined here — every call goes directly into the IsoME package
(non-exported functions are accessed as `IsoME.<function>`). The cells mirror the body of
`RealAxisSolver()` and `findTc_RealAxis()` in `src/realAxisSolver.jl`, just without the try/catch wrappers.

In [ ]:
# ----- Environment setup -----
# Assumes the notebook kernel is started inside the scripts/ folder (default in Jupyter/VSCode),
# so the package root is one level up.
import Pkg
pkg_dir = abspath(joinpath(@__DIR__, ".."))
Pkg.activate(pkg_dir)

using IsoME
using Plots

## Step 1) Read in of the inputs

Define the input struct `arguments` exactly as you would for `RealAxisSolver(inp)`.
The example below is the Nb test case shipped with the package (cDOS mode, a2F file only).
For a vDOS calculation set `cDOS_flag = 0` and provide `dos_file` (and `Weep_file` for vDOS+W).

In [ ]:
inp = arguments(
    a2f_file    = joinpath(pkg_dir, "test", "Nb", "Nb.a2f"),
    outdir      = joinpath(@__DIR__, "output"),
    cDOS_flag   = 1,
    temps       = [6.0],
    omega_c     = 2000.0,
    domega      = 0.4,
    mixing_beta = 0.5,
    mu          = 0.38,
    typEl       = 10000.0,
    conv_thr    = 1e-3,
    ind_smear   = 15,
);

In [ ]:
# Create the output directory + log file, validate the inputs and read the files
# (same sequence as at the top of RealAxisSolver).
# Note: createDirectory! appends _1, _2, ... if outdir already exists.
strIsoME = IsoME.printIsoME()
log_file, errorLogger = IsoME.createDirectory!(inp, strIsoME)

IsoME.checkInput!(inp, realSolver=true)

console, matval, ML_Tc = IsoME.InputParser!(inp, log_file, mode=1)

IsoME.printFlagsAsText(inp, matval, log_file, mode="realFreq")

# unpack the material-specific values for inspection
(a2f_omega, a2f, dos_en, dos, Weep, dosef, idx_ef, ndos, BCS_gap) = matval;

In [ ]:
# --- Debug: inspect the read-in quantities ---
println("ML_Tc estimate : ", round(ML_Tc, digits=2), " K")
println("BCS gap        : ", round(BCS_gap, digits=3), " meV")
println("μ*_ME          : ", inp.muc_ME)
println("μ*_AD          : ", inp.muc_AD)

p1 = plot(a2f_omega, a2f, xlabel="ω / meV", ylabel="α²F(ω)", label="", title="Eliashberg spectral function")
if inp.cDOS_flag == 0
    p2 = plot(dos_en, dos, xlabel="ε / meV", ylabel="N(ε)", label="", title="DOS")
    plot(p1, p2, layout=(1, 2), size=(900, 350))
else
    p1
end

## Step 2) Calculation of the kernels

`IsoME.precompute` sets up the ω-axes and computes the (complex) phonon kernels
K⁺(ω,ω′) and K⁻(ω,ω′) at a given temperature.

**Note:** the kernels depend on temperature through β = 1/(k_B·T), so this step has to be
repeated for every temperature (as done in the loop of step 3b). Here we compute them for a
single temperature so they can be inspected in isolation.

In [ ]:
itemp = inp.temps[1]            # temperature in K, pick any value to debug
β = 1 / (IsoME.kb * itemp)

realAxisParameter = IsoME.precompute(β, inp, matval, console, log_file)

# w_static : ω-grid of Z(ω), Δ(ω) and χ(ω) - all three share this grid
# kernel   : PhononKernel, the A/B tables the ω′-integral is assembled from
(w_static, kernel) = realAxisParameter

# K⁺/K⁻ are not stored as a matrix and not as callables: the ω′-integral assembles them
# on the fly from A and B (see tail_accumulate! in src/Kernels.jl). Same assembly here, so
# the cells below can plot them:
#     𝒦(ω, ω′) = A(x) + f(ω′)·B(x),  x = ω′ - ω        (difference channel)
#     𝒦(ω,-ω′) = A(x) + f(-ω′)·B(x), x = -ω - ω′       (sum channel)
#     K⁺ = 𝒦(ω,ω′) - 𝒦(ω,-ω′)        K⁻ = -𝒦(ω,ω′) - 𝒦(ω,-ω′)
function K_pm(ker, w, wp, β)
    AB(x) = IsoME._lerpAB(ker.A, ker.B, (x - ker.x_min) / ker.dx + 1, ker.n_x)
    fp = 1 / (exp(β * wp) + 1)
    a_d, b_d = AB(wp - w);  K_dif = a_d + fp * b_d
    a_s, b_s = AB(-w - wp); K_sum = a_s + (1 - fp) * b_s
    return (K_dif - K_sum, -K_dif - K_sum)       # (K⁺, K⁻)
end
Kp_func(w, wp) = K_pm(kernel, w, wp, β)[1]
Km_func(w, wp) = K_pm(kernel, w, wp, β)[2];

In [ ]:
# --- Debug: visualize the kernels ---
w_dbg = range(first(w_static), last(w_static), length=300)

hm_p = heatmap(w_dbg, w_dbg, [real(Kp_func(w, wp)) for wp in w_dbg, w in w_dbg],
               xlabel="ω / meV", ylabel="ω′ / meV", title="Re K⁺(ω,ω′)")
hm_m = heatmap(w_dbg, w_dbg, [real(Km_func(w, wp)) for wp in w_dbg, w in w_dbg],
               xlabel="ω / meV", ylabel="ω′ / meV", title="Re K⁻(ω,ω′)")
plot(hm_p, hm_m, layout=(1, 2), size=(950, 380))

In [ ]:
# --- Debug: kernel slices at fixed ω ---
w_fix = w_static[1]
plot(w_dbg, [real(Kp_func(w_fix, wp)) for wp in w_dbg], label="Re K⁺")
plot!(w_dbg, [imag(Kp_func(w_fix, wp)) for wp in w_dbg], label="Im K⁺")
plot!(w_dbg, [real(Km_func(w_fix, wp)) for wp in w_dbg], label="Re K⁻")
plot!(w_dbg, [imag(Km_func(w_fix, wp)) for wp in w_dbg], label="Im K⁻",
      xlabel="ω′ / meV", title="K(ω=$(round(w_fix, digits=2)) meV, ω′)")

## Step 3) Evaluation of the Migdal-Eliashberg equations on the real axis

### 3a) Single temperature

Solve at the temperature `itemp` used for the kernels above. This is the same dispatch as in
`findTc_RealAxis`: cDOS → `solve_realAxis_cDOS`, vDOS → cDOS run as initial guess
(`initialize_real_axis_vDOS`) followed by `solve_realAxis_vDOS`.

In [ ]:
if inp.cDOS_flag == 1
    data, state = IsoME.solve_realAxis_cDOS(itemp, inp, console.cDOS, matval, realAxisParameter, log_file)
else
    state0 = IsoME.initialize_real_axis_vDOS(itemp, inp, console.cDOS, matval, realAxisParameter, log_file)
    data, state = IsoME.solve_realAxis_vDOS(itemp, inp, console.vDOS, matval, realAxisParameter, state0, log_file)
end

println("\nZ(ω→0)  = ", data[1])
println("Δ(ω→0)  = ", data[2], " meV")
inp.cDOS_flag == 0 && println("χ(ω→0)  = ", data[3], " meV")

In [ ]:
# --- Debug: the converged self-energy components on the real axis ---
pd = plot(w_static, real(state.delta), label="Re Δ(ω)")
plot!(pd, w_static, imag(state.delta), label="Im Δ(ω)", xlabel="ω / meV", ylabel="Δ / meV")

pz = plot(w_static, real(state.Z), label="Re Z(ω)")
plot!(pz, w_static, imag(state.Z), label="Im Z(ω)", xlabel="ω / meV", ylabel="Z")

if inp.cDOS_flag == 0
    pc = plot(w_static, real(state.chi), label="Re χ(ω)")
    plot!(pc, w_static, imag(state.chi), label="Im χ(ω)", xlabel="ω / meV", ylabel="χ / meV")
    plot(pd, pz, pc, layout=(1, 3), size=(1200, 350))
else
    plot(pd, pz, layout=(1, 2), size=(900, 350))
end

### 3b) Loop over all temperatures (optional)

This cell repeats steps 2 + 3 for every temperature in `inp.temps`, mirroring the
given-temperature branch of `findTc_RealAxis` (kernels are recomputed per temperature;
in vDOS mode the converged state of the previous temperature is reused as the initial guess).

In [ ]:
temps   = sort(inp.temps)
Znorm0  = ComplexF64[]
Delta0  = ComplexF64[]
Shift0  = ComplexF64[]
Tc      = [NaN, NaN]
realAxisState = nothing

for (iT, T) in enumerate(temps)
    β_T = 1 / (IsoME.kb * T)
    par_T = IsoME.precompute(β_T, inp, matval, console, log_file)   # step 2

    if inp.cDOS_flag == 0 && iT == 1
        realAxisState = IsoME.initialize_real_axis_vDOS(T, inp, console.cDOS, matval, par_T, log_file)
    end

    # step 3
    if inp.cDOS_flag == 0
        data_T, realAxisState = IsoME.solve_realAxis_vDOS(T, inp, console.vDOS, matval, par_T, realAxisState, log_file)
        push!(Shift0, data_T[3])
    else
        data_T, realAxisState = IsoME.solve_realAxis_cDOS(T, inp, console.cDOS, matval, par_T, log_file)
    end
    push!(Znorm0, data_T[1])
    push!(Delta0, data_T[2])

    # escape once the gap has closed (same logic as findTc_RealAxis)
    if isnan(real(data_T[2]))
        length(Delta0) > 1 && (Tc[1] = temps[iT-1])
        Tc[2] = T
        break
    end
end

all(.~isnan.(real.(Delta0))) && (Tc[1] = temps[length(Delta0)])

IsoME.printSummary(inp, Tc, log_file)

In [ ]:
# --- Debug: gap vs. temperature ---
nconv = length(Delta0)
scatter(temps[1:nconv], real.(Delta0), xlabel="T / K", ylabel="Re Δ(0) / meV",
        label="Δ(ω→0)", title="Superconducting gap vs. temperature")

In [ ]:
# ----- Finish: flush and close the log file -----
inp.testMode || close(log_file)